# Gold test Few-shot Cosine (Prompt 1)

Génère le dataset de test few-shot en sélectionnant les exemples d'entraînement les plus proches de chaque instance de test via **similarité cosine** (sentence-transformers). Pour chaque instance de test on garde **1 exemple Entailment** et **1 exemple Contradiction** (les plus proches pour chaque label).

## 1. Chemins et configuration

In [7]:
import json
from pathlib import Path

cwd = Path(".").resolve()
if (cwd / "Gold_test.json").exists():
    NLI4CT_ROOT = cwd
elif (cwd.parent.parent / "Gold_test.json").exists():
    NLI4CT_ROOT = cwd.parent.parent
elif (cwd.parent / "Gold_test.json").exists():
    NLI4CT_ROOT = cwd.parent
else:
    NLI4CT_ROOT = cwd

GOLD_TEST_JSON = NLI4CT_ROOT / "Gold_test.json"
TRAIN_JSON = NLI4CT_ROOT / "train.json"
CT_JSON_DIR = NLI4CT_ROOT / "CT_json"
OUT_DIR = NLI4CT_ROOT / "results" / "Fewshot_Cosine"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1 Entailment + 1 Contradiction (les plus proches en similarité cosine pour chaque label)
N_PER_LABEL = 1
SENTENCE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # léger et rapide

print("NLI4CT_ROOT:", NLI4CT_ROOT)
print("OUT_DIR:", OUT_DIR)

NLI4CT_ROOT: /Users/lubin/Documents/NLI_Finetuning/NLI4CT
OUT_DIR: /Users/lubin/Documents/NLI_Finetuning/NLI4CT/results/Fewshot_Cosine


## 2. Fonctions de formatage (Prompt 1)

In [8]:
def load_ct_data(ct_id):
    ct_file = CT_JSON_DIR / f"{ct_id}.json"
    if not ct_file.exists():
        return None
    with open(ct_file, "r", encoding="utf-8") as f:
        return json.load(f)


def extract_evidence(ct_data, section_id, evidence_indices):
    if ct_data is None:
        return ""
    section_data = ct_data.get(section_id, [])
    if not section_data:
        return ""
    return "\n".join(section_data[i] for i in evidence_indices if 0 <= i < len(section_data))


SYSTEM_MSG_FEWSHOT = "You will see several examples of premise-hypothesis pairs with their classification (Entailment or Contradiction). Use these examples to understand the task. Then classify the relationship between the last premise and hypothesis. Respond with only one word: 'Entailment' or 'Contradiction'."


def format_input_text_prompt1(statement, primary_evidence, secondary_evidence=None):
    premise_content = f"{primary_evidence}\n\n{secondary_evidence}" if secondary_evidence else primary_evidence
    return f"PREMISE: {premise_content}\n\nHYPOTHESIS: {statement}"


def process_json_to_examples(input_path, format_fn):
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    examples = []
    for entry_id, entry in data.items():
        try:
            statement = entry.get("Statement", "")
            label = entry.get("Label", "")
            section_id = entry.get("Section_id", "")
            primary_id = entry.get("Primary_id", "")
            entry_type = entry.get("Type", "Single")
            primary_ct_data = load_ct_data(primary_id)
            if primary_ct_data is None:
                continue
            primary_evidence = extract_evidence(primary_ct_data, section_id, entry.get("Primary_evidence_index", []))
            secondary_evidence = None
            if entry_type == "Comparison":
                secondary_ct_data = load_ct_data(entry.get("Secondary_id", ""))
                if secondary_ct_data is None:
                    continue
                secondary_evidence = extract_evidence(secondary_ct_data, section_id, entry.get("Secondary_evidence_index", []))
            user_content = format_fn(statement, primary_evidence, secondary_evidence)
            examples.append({"user_content": user_content, "label": label, "type": entry_type, "section_id": section_id})
        except Exception:
            continue
    return examples


def make_fewshot_messages(test_ex, shot_examples, use_system_message=True):
    messages = []
    if use_system_message:
        messages.append({"role": "system", "content": SYSTEM_MSG_FEWSHOT})
    for shot in shot_examples:
        messages.append({"role": "user", "content": shot["user_content"]})
        messages.append({"role": "assistant", "content": shot["label"]})
    messages.append({"role": "user", "content": test_ex["user_content"]})
    messages.append({"role": "assistant", "content": test_ex["label"]})
    return messages

## 3. Embeddings + similarité cosine (sentence-transformers)

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

encoder = SentenceTransformer(SENTENCE_MODEL)


def embed_texts(texts, batch_size=32):
    return encoder.encode(texts, batch_size=batch_size, show_progress_bar=True)


def embed_examples(examples):
    texts = [ex["user_content"] for ex in examples]
    embs = embed_texts(texts)
    for ex, emb in zip(examples, embs):
        ex["embedding"] = emb
    return examples


def select_cosine_shots(test_embedding, train_examples_with_emb, n_per_label=1):
    """Sélectionne les n_per_label exemples les plus proches par label (Entailment, Contradiction)."""
    embeddings = np.array([ex["embedding"] for ex in train_examples_with_emb])
    sims = cosine_similarity([test_embedding], embeddings)[0]
    shots = []
    for label in ["Entailment", "Contradiction"]:
        indices = [i for i, ex in enumerate(train_examples_with_emb) if ex["label"] == label]
        if not indices:
            continue
        best_idx = indices[np.argmax(sims[indices])]
        shots.append(train_examples_with_emb[best_idx])
    return shots

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2195.84it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 4. Chargement train / test et embedding du train

In [10]:
train_examples = process_json_to_examples(TRAIN_JSON, format_input_text_prompt1)
test_examples = process_json_to_examples(GOLD_TEST_JSON, format_input_text_prompt1)
print("Train:", len(train_examples), "| Test:", len(test_examples))

train_with_emb = embed_examples(train_examples)

Train: 1700 | Test: 500


Batches: 100%|██████████| 54/54 [00:08<00:00,  6.33it/s]


## 5. Génération du JSONL few-shot Cosine (Prompt 1)

In [11]:
out_path = OUT_DIR / "Gold_test_fewshot_cosine_prompt1.jsonl"
count = 0
with open(out_path, "w", encoding="utf-8") as f:
    for test_ex in tqdm(test_examples, desc="Building few-shot JSONL"):
        test_emb = encoder.encode([test_ex["user_content"]])[0]
        shots = select_cosine_shots(test_emb, train_with_emb, n_per_label=N_PER_LABEL)
        messages = make_fewshot_messages(test_ex, shots, use_system_message=True)
        f.write(json.dumps({"messages": messages}, ensure_ascii=False) + "\n")
        count += 1
print(f"Gold_test_fewshot_cosine_prompt1.jsonl : {count} lignes -> {out_path}")

Building few-shot JSONL: 100%|██████████| 500/500 [00:09<00:00, 53.68it/s]

Gold_test_fewshot_cosine_prompt1.jsonl : 500 lignes -> /Users/lubin/Documents/NLI_Finetuning/NLI4CT/results/Fewshot_Cosine/Gold_test_fewshot_cosine_prompt1.jsonl


## 6. Aperçu d'une entrée

In [12]:
with open(out_path, "r", encoding="utf-8") as f:
    first = json.loads(f.readline())
msgs = first["messages"]
print("Nombre de messages:", len(msgs))
print("Rôles:", [m["role"] for m in msgs])
print("Dernier user (début):", msgs[-2]["content"][:200], "...")
print("Dernier assistant (gold):", msgs[-1]["content"])

Nombre de messages: 7
Rôles: ['system', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant']
Dernier user (début): PREMISE: Exclusion Criteria:
  History of bilateral mastectomy, osteoporosis or renal impairment.

Exclusion Criteria:
  Severe claustrophobia

HYPOTHESIS: Women suffering from both claustrophobia and ...
Dernier assistant (gold): Contradiction
